## SVOD_TEMPLATE_CREATION

###📍IMPORTANT: Always check the order of WBTV and Foundry data

In [ ]:
# ============================================================
# 📚 LIBRARIES - EXTERNAL
# ============================================================
import os
import time
import warnings
import logging

import pandas as pd
from openpyxl import load_workbook

warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_NO_TF"] = "1"

# ============================================================
# LOGGING
# ============================================================
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

if not logger.handlers:
    formatter = logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    )
    file_handler = logging.FileHandler("SVOD.log", mode="w")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

# ============================================================
# 📚 LIBRARIES - OWN FUNCTIONS
# ============================================================
from Packages import matching_pipeline

pipeline = matching_pipeline()

# ============================================================
# INPUTS
# ============================================================
logger.info("Enter short and long synopsis word count thresholds")

while True:
    try:
        short = int(input("Enter SHORT synopsis limit: ").strip())
        long = int(input("Enter LONG synopsis limit: ").strip())
        too_long = int(input("Enter TOO LONG synopsis limit: ").strip())

        if short <= 0 or long <= 0 or too_long <= 0:
            logger.error("Values must be positive integers. Try again.")
            continue

        break

    except ValueError:
        logger.error("Invalid input. Please enter numeric values.")

template_type = pipeline.ask_content_type()

final_df = pipeline.run_template_pipeline(
    content_type=template_type,
    top_k=5,
    ce_threshold=0.75
)

output_folder = pipeline.select_output_folder()
format_check_path = os.path.join(output_folder, "WBTVD or WB2B or FOUNDRY - Format.csv")
english_path = format_check_path
temp_path = output_folder

final_df.to_csv(format_check_path, index=False)

logger.info("File verification Done")
verify = input("File verification Done [y/n]: ").strip().lower()

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def prepare_english_dataframe(df, template_type, short, long, too_long):
    df = df.copy()

    if "Season" not in df.columns:
        df["Season"] = 0
    if "Episode" not in df.columns:
        df["Episode"] = 0

    df["Season"] = pd.to_numeric(df["Season"], errors="coerce").fillna(0).astype(int)
    df["Episode"] = pd.to_numeric(df["Episode"], errors="coerce").fillna(0).astype(int)

    if "Primary Release Date" in df.columns:
        df["Primary Release Date"] = pd.to_datetime(df["Primary Release Date"], errors="coerce")

    if template_type == "series":
        df = df.sort_values(["Season", "Episode"]).reset_index(drop=True)
    else:
        title_col = pipeline.get_output_title_column(df)
        if title_col:
            df = df.sort_values(by=[title_col]).reset_index(drop=True)
        else:
            df = df.reset_index(drop=True)

    df["Sr.No."] = range(1, len(df) + 1)
    df["Category"] = "WB"

    title_col = pipeline.get_output_title_column(df)
    if title_col is None:
        raise ValueError("❌ No valid title column found for output creation.")

    df["Source Title (Long Description)"] = df[title_col]

    if template_type == "series":
        df["Source Title (Long Description)"] = [
            f"{t}: Season {s}" if e == 0 and s != 0 else t
            for t, e, s in zip(
                df["Source Title (Long Description)"],
                df["Episode"],
                df["Season"]
            )
        ]

    df["Localized Title"] = ""

    if "MPM Number" in df.columns:
        df["WM Internal Reference"] = df["MPM Number"]
    elif "uuid" in df.columns:
        df["WM Internal Reference"] = df["uuid"]
    else:
        df["WM Internal Reference"] = ""

    if "Primary Release Date" in df.columns:
        df["US Release Date"] = df["Primary Release Date"].dt.strftime("%Y-%m-%d")
    else:
        df["US Release Date"] = ""

    # Synopsis columns
    short_src_col = f"Synopsis (Short) SOURCE ({short} Character Limit)"
    short_trans_col = f"Synopsis (Short) TRANSLATION ({short} Character Limit)"
    long_src_col = f"Synopsis (Long) SOURCE DATA ({long} Character Limit)"
    long_trans_col = f"Synopsis (Long) TRANSLATION ({long} Character Limit)"
    too_long_src_col = f"Synopsis (Long) SOURCE DATA ({too_long} Character Limit)"
    too_long_trans_col = f"Synopsis (Long) TRANSLATION ({too_long} Character Limit)"

    df[short_src_col] = pipeline.char_limit(short, df, "yes")
    df[short_trans_col] = ""

    df[long_src_col] = pipeline.char_limit(long, df, "yes")
    df[long_trans_col] = ""

    df[too_long_src_col] = pipeline.char_limit(too_long, df, "yes")
    df[too_long_trans_col] = ""

    # Character counter columns (unique names)
    row_count = len(df)
    df[f"Source char. Counter ({short})"] = [f"=LEN(I{i})" for i in range(2, row_count + 2)]
    df[f"Translation char. Count ({short})"] = [f"=LEN(K{i})" for i in range(2, row_count + 2)]
    df[f"Source char. Counter ({long})"] = [f"=LEN(M{i})" for i in range(2, row_count + 2)]
    df[f"Translation char. Count ({long})"] = [f"=LEN(O{i})" for i in range(2, row_count + 2)]
    df[f"Source char. Counter ({too_long})"] = [f"=LEN(Q{i})" for i in range(2, row_count + 2)]
    df[f"Translation char. Count ({too_long})"] = [f"=LEN(S{i})" for i in range(2, row_count + 2)]

    final_columns = [
        "Sr.No.", "Category", "Season", "Episode",
        "Source Title (Long Description)", "Localized Title",
        "WM Internal Reference", "US Release Date",
        short_src_col, f"Source char. Counter ({short})",
        short_trans_col, f"Translation char. Count ({short})",
        long_src_col, f"Source char. Counter ({long})",
        long_trans_col, f"Translation char. Count ({long})",
        too_long_src_col, f"Source char. Counter ({too_long})",
        too_long_trans_col, f"Translation char. Count ({too_long})"
    ]

    for col in final_columns:
        if col not in df.columns:
            df[col] = ""

    return df[final_columns]


def prepare_translation_dataframe(df_trans, english_df, template_type, short, long, too_long, lang_name):
    df_trans = df_trans.copy()
    english_df = english_df.copy()

    # Ensure required columns exist
    if "season-number" in df_trans.columns:
        df_trans["Season"] = pd.to_numeric(df_trans["season-number"], errors="coerce").fillna(0).astype(int)
    else:
        df_trans["Season"] = 0

    if "episode-number" in df_trans.columns:
        df_trans["Episode"] = pd.to_numeric(df_trans["episode-number"], errors="coerce").fillna(0).astype(int)
    else:
        df_trans["Episode"] = 0

    df_trans = df_trans.sort_values(["Season", "Episode"]).reset_index(drop=True)

    if "Primary Release Date" in english_df.columns:
        english_df["Primary Release Date"] = pd.to_datetime(
            english_df["Primary Release Date"], errors="coerce"
        )

    required_english_cols = ["uuid", "Season", "Episode", "Primary Release Date"]
    optional_english_cols = []

    if "*Title name" in english_df.columns:
        optional_english_cols.append("*Title name")
    if "MPM Number" in english_df.columns:
        optional_english_cols.append("MPM Number")

    merge_cols = required_english_cols + optional_english_cols

    df = pd.merge(
        english_df[merge_cols],
        df_trans,
        on="uuid",
        how="left",
        suffixes=("_eng", "_trans")
    )

    if "*Title name" in df.columns and df["*Title name"].isna().any():
        raise ValueError(f"❌ English mismatch in {lang_name}")

    df["Sr.No."] = range(1, len(df) + 1)
    df["Category"] = "WB"

    if "Season_eng" in df.columns:
        df["Season"] = df["Season_eng"]
    if "Episode_eng" in df.columns:
        df["Episode"] = df["Episode_eng"]

    title_col = pipeline.get_output_title_column(df)
    if title_col is None:
        if "*Title name" in df.columns:
            title_col = "*Title name"
        else:
            raise ValueError(f"❌ No title column found in translated sheet: {lang_name}")

    df["Source Title (Long Description)"] = df[title_col]

    if "main-title" in df.columns:
        df["Localized Title"] = df["main-title"].fillna("")
    else:
        df["Localized Title"] = ""

    if template_type == "series":
        df["Source Title (Long Description)"] = [
            f"{i}: Season {int(s)}" if j == 0 and s != 0 else i
            for i, j, s in zip(df["Source Title (Long Description)"], df["Episode"], df["Season"])
        ]

    if "MPM Number" in df.columns:
        df["WM Internal Reference"] = df["MPM Number"]
    else:
        df["WM Internal Reference"] = df["uuid"]

    if "Primary Release Date" in df.columns:
        df["US Release Date"] = pd.to_datetime(
            df["Primary Release Date"], errors="coerce"
        ).dt.strftime("%Y-%m-%d")
    else:
        df["US Release Date"] = ""

    # Synopsis columns
    short_src_col = f"Synopsis (Short) SOURCE ({short} Character Limit)"
    short_trans_col = f"Synopsis (Short) TRANSLATION ({short} Character Limit)"
    long_src_col = f"Synopsis (Long) SOURCE DATA ({long} Character Limit)"
    long_trans_col = f"Synopsis (Long) TRANSLATION ({long} Character Limit)"
    too_long_src_col = f"Synopsis (Long) SOURCE DATA ({too_long} Character Limit)"
    too_long_trans_col = f"Synopsis (Long) TRANSLATION ({too_long} Character Limit)"

    df[short_src_col] = pipeline.char_limit(short, english_df, "yes")
    df[short_trans_col] = pipeline.char_limit(short, df)

    df[long_src_col] = pipeline.char_limit(long, english_df, "yes")
    df[long_trans_col] = pipeline.char_limit(long, df)

    df[too_long_src_col] = pipeline.char_limit(too_long, english_df, "yes")
    df[too_long_trans_col] = pipeline.char_limit(too_long, df)

    row_count = len(df)
    df[f"Source char. Counter ({short})"] = [f"=LEN(I{i})" for i in range(2, row_count + 2)]
    df[f"Translation char. Count ({short})"] = [f"=LEN(K{i})" for i in range(2, row_count + 2)]
    df[f"Source char. Counter ({long})"] = [f"=LEN(M{i})" for i in range(2, row_count + 2)]
    df[f"Translation char. Count ({long})"] = [f"=LEN(O{i})" for i in range(2, row_count + 2)]
    df[f"Source char. Counter ({too_long})"] = [f"=LEN(Q{i})" for i in range(2, row_count + 2)]
    df[f"Translation char. Count ({too_long})"] = [f"=LEN(S{i})" for i in range(2, row_count + 2)]

    final_columns = [
        "Sr.No.", "Category", "Season", "Episode",
        "Source Title (Long Description)", "Localized Title",
        "WM Internal Reference", "US Release Date",
        short_src_col, f"Source char. Counter ({short})",
        short_trans_col, f"Translation char. Count ({short})",
        long_src_col, f"Source char. Counter ({long})",
        long_trans_col, f"Translation char. Count ({long})",
        too_long_src_col, f"Source char. Counter ({too_long})",
        too_long_trans_col, f"Translation char. Count ({too_long})"
    ]

    for col in final_columns:
        if col not in df.columns:
            df[col] = ""

    return df[final_columns]


# ============================================================
# MAIN TEMPLATE CREATION
# ============================================================
if verify == "y":
    only_english = input("Is the multilanguage folder available? [y/n]: ").strip().lower()
    path = None

    if only_english == "y":
        path = pipeline.pick_multilang_folder()

    series_name = input("Enter the Series name: ").strip()
    series = f"SVOD_{series_name}"
    en_path = english_path
    temp = temp_path

    output_file = os.path.join(temp, f"{series}_Template.xlsx")

    # =========================
    # DELETE OLD FILE
    # =========================
    if os.path.exists(output_file):
        try:
            os.remove(output_file)
        except PermissionError:
            raise RuntimeError("❌ Excel file is open. Close it and retry.")

    # =========================
    # LOAD ENGLISH MASTER
    # =========================
    english = pd.read_csv(en_path)
    english_prepared = prepare_english_dataframe(english, template_type, short, long, too_long)

    written_sheets = []

    try:
        with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
            # ====================================
            # ENGLISH ONLY
            # ====================================
            if only_english == "n":
                english_prepared.to_excel(writer, sheet_name="English", index=False)
                written_sheets.append("English")

            # ====================================
            # TRANSLATION FILES
            # ====================================
            else:
                english_prepared.to_excel(writer, sheet_name="English", index=False)
                written_sheets.append("English")

                for file in os.listdir(path):
                    if not file.endswith(".csv"):
                        continue

                    try:
                        parts = file.split("-")
                        if len(parts) < 3 or "export_" not in parts[2]:
                            logger.warning(f"Skipping file with unexpected format: {file}")
                            continue

                        name = parts[2].split("export_")[1].replace("_", " ").replace(".csv", "").strip()

                        if name == "English United States":
                            continue

                        df_trans = pd.read_csv(os.path.join(path, file))

                        translated_df = prepare_translation_dataframe(
                            df_trans=df_trans,
                            english_df=english,
                            template_type=template_type,
                            short=short,
                            long=long,
                            too_long=too_long,
                            lang_name=name
                        )

                        safe_sheet_name = name[:31]  # Excel sheet limit
                        translated_df.to_excel(writer, sheet_name=safe_sheet_name, index=False)
                        written_sheets.append(safe_sheet_name)

                    except Exception as e:
                        logger.error(f"Failed processing {file}: {e}")

                if len(written_sheets) == 1:  # only English written
                    pd.DataFrame({"Message": ["No valid translation data found"]}).to_excel(
                        writer, sheet_name="Info", index=False
                    )

    except PermissionError:
        raise RuntimeError("❌ Excel file is open. Close it before running.")

    time.sleep(2)

    # ============================================================
    # SUMMARY SHEET
    # ============================================================
    summary_rows = []

    sheets = pd.read_excel(output_file, sheet_name=None)

    for sheet_name, df in sheets.items():
        if sheet_name == "Info":
            continue

        cols = df.columns.tolist()

        # Defaults
        col9_wc = 0
        col13_wc = 0
        col17_wc = 0

        col9_status = col13_status = col17_status = "Not available"
        col11_status = col15_status = col19_status = "Not available"

        col11_len_status = col15_len_status = col19_len_status = "Not available"

        # =========================
        # COLUMN 11 (Short Translation)
        # =========================
        if len(cols) >= 11:
            col11 = df.iloc[:, 10]
            col11_status = pipeline.availability_status(col11)
            col11_len_status = pipeline.synopsis_status(col11, short)

        # =========================
        # COLUMN 15 (Long Translation)
        # =========================
        if len(cols) >= 15:
            col15 = df.iloc[:, 14]
            col15_status = pipeline.availability_status(col15)
            col15_len_status = pipeline.synopsis_status(col15, long)

        # =========================
        # COLUMN 19 (800 Translation)
        # =========================
        if len(cols) >= 19:
            col19 = df.iloc[:, 18]
            col19_status = pipeline.availability_status(col19)
            col19_len_status = pipeline.synopsis_status(col19, too_long)

        # =========================
        # COLUMN 9 (Short English)
        # =========================
        if len(cols) >= 11:
            col9 = df.iloc[:, 8]
            col9_status = pipeline.synopsis_status(col9, short)
            col9_wc = pipeline.conditional_word_count(col9, col11)

        # =========================
        # COLUMN 13 (Long English)
        # =========================
        if len(cols) >= 15:
            col13 = df.iloc[:, 12]
            col13_status = pipeline.synopsis_status(col13, long)
            col13_wc = pipeline.conditional_word_count(col13, col15)

        # =========================
        # COLUMN 17 (800 English)
        # =========================
        if len(cols) >= 19:
            col17 = df.iloc[:, 16]
            col17_status = pipeline.synopsis_status(col17, too_long)
            col17_wc = pipeline.conditional_word_count(col17, col19)

        # =========================
        # APPEND SUMMARY
        # =========================
        summary_rows.append({
            "Sheet": sheet_name,

            f"Synopsis (Short) TRANSLATION ({short} Character Limit)": col11_status,
            f"Synopsis (Long) TRANSLATION ({long} Character Limit)": col15_status,
            f"Synopsis (800) TRANSLATION ({too_long} Character Limit)": col19_status,

            f"Synopsis (Short) ENGLISH ({short} Character Limit)": col9_status,
            f"Synopsis (Long) ENGLISH ({long} Character Limit)": col13_status,
            f"Synopsis (800) ENGLISH ({too_long} Character Limit)": col17_status,

            f"Synopsis (Short) TRANS_len ({short} Character Limit)": col11_len_status,
            f"Synopsis (Long) TRANS_len ({long} Character Limit)": col15_len_status,
            f"Synopsis (800) TRANS_len ({too_long} Character Limit)": col19_len_status,

            "Word_Count": col9_wc + col13_wc + col17_wc
        })
    summary_df = pd.DataFrame(summary_rows)

    with pd.ExcelWriter(output_file, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
        summary_df.to_excel(writer, sheet_name="Info", index=False)

    wb = load_workbook(output_file)

    for name in written_sheets:
        if name in wb.sheetnames:
            ws = wb[name]
            pipeline.format_sheet_openpyxl(ws, short, long, too_long)

    wb.save(output_file)

    logger.info(f"Template '{series}' created successfully.")
    print(f"✅ Template '{series}' created successfully at:\n{output_file}")

Select folder to save output files...
✅ Folder cleaned successfully
Selected Output Folder: C:/Users/mshanmugam/OneDrive - Warner Bros. Discovery/JUPYTER_PY/AI Projects/Automated Metadata Template Creation (SVOD)/Output_folder
✅ Template 'SVOD_LORT_Extra' created successfully at:
C:/Users/mshanmugam/OneDrive - Warner Bros. Discovery/JUPYTER_PY/AI Projects/Automated Metadata Template Creation (SVOD)/Output_folder\SVOD_LORT_Extra_Template.xlsx
